# EDA
**EXPLORATORY DATA ANALYSIS & DISCOVERY**

---


## Inhalt

- [Data Acquisition](#data-acquisition)
  - [Preparation](#preparation)
    - [Imports](#imports)
    - [Settings](#settings)
    - [Constants](#constants)
  - [Data Gathering](#data-gathering)
    - [Note on Large Dataset Size](#note-on-large-dataset-size)
- [Analysis](#analysis)
  - [Basic Statistical Analysis](#basic-statistical-analysis)
  - [Data Completeness](#data-completeness)
    - [Missings — Schedule vs. Delay Asymmetrie](#missings-schedule-vs-delay-asymmetrie)
    - [Completeness Lesson: `is_windy` — 100% NaN across all 94M Rows](#completeness-lesson-iswindy-100-nan-across-all-94m-rows)
  - [Data Integrity](#data-integrity)
    - [Extreme Delay-Werte (-29.892s / +34.685s)](#extreme-delay-werte-29892s-34685s)
    - [BPUIC Outlier (max 859,600,701)](#bpuic-outlier-max-859600701)
    - [Meteo Gaps (Missing Measurements, Rolling Window)](#meteo-gaps-missing-measurements-rolling-window)
    - [Humidity > 100% (Sensor-Kalibrierungsfehler)](#humidity-100-sensor-kalibrierungsfehler)
    - [Canceled Trips (~4.5% der Fahrten)](#canceled-trips-45-der-fahrten)
  - [Data Distribution](#data-distribution)
    - [Numerische Features — Rohe Verteilung](#numerische-features-rohe-verteilung)
    - [Kategorische Features — Verteilung](#kategorische-features-verteilung)
    - [Beurteilung der Verteilungen](#beurteilung-der-verteilungen)
  - [Data Relationships](#data-relationships)
    - [Korrelations-Findings](#korrelations-findings)
  - [Outlier Detection](#outlier-detection)
    - [Delay-Features — Outlier Detail (IQR 1.5× und 3×)](#delay-features-outlier-detail-iqr-15-und-3)
    - [Meteo-Features — Outlier Detail (precipitation, wind_speed, temperature)](#meteo-features-outlier-detail-precipitation-windspeed-temperature)
    - [Outlier-Findings](#outlier-findings)
  - [Features Inspection](#features-inspection)
    - [Feature Ideas — Overview](#feature-ideas-overview)
- [Key Findings & Next Steps](#key-findings-next-steps)
  - [Konsolidierte Findings — nach Topic](#konsolidierte-findings-nach-topic)
  - [Offene Fragen — alle in der EDA beantwortet](#offene-fragen-alle-in-der-eda-beantwortet)
  - [Prognose der Datensatz-Reduktion nach dem Cleaning](#prognose-der-datensatz-reduktion-nach-dem-cleaning)


## Data Acquisition



### Preparation




#### Imports

In [ ]:
# data
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# admin
from pathlib import Path
import psutil
import sys
import os

# wgnd
from wgnd.core.theme import setup
from wgnd.core._output import (
    section_header, 
    log, 
    success, 
    warn
)
from wgnd.inspect import (
    inspect,
    inspect_missing,
    inspect_outliers,
    inspect_outlier_detail,
    inspect_correlations,
)
from zh_tram_flow.utils_polars import get_categorical_stats

setup()

#### Settings

In [ ]:
%load_ext autoreload
%autoreload 2

#### Constants

In [ ]:
BASE_DIR        = Path('../')
DATA_RAW_DIR    = BASE_DIR / 'data' / 'raw'
DATA            = DATA_RAW_DIR / 'zh-tram-data-master.parquet'

SEED            = 42

print(DATA)

### Data Gathering

In [ ]:

# RAM Test Polars Eager vs Lazy

def get_ram():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

print(f"RAM: {get_ram():.2f} MB")
available_ram = psutil.virtual_memory().available / (1024**3) # in GB
print(f"Freier RAM: {available_ram:.2f} GB")

#print()
#df_raw = pl.read_parquet(DATA)
#print(f"Nach Eager Read: {get_ram():.2f} MB")

print()
lf_raw = pl.scan_parquet(DATA)
print(f"Nach Lazy Scan: {get_ram():.2f} MB")
print('Lazy Scan, keine Veränderung')

print()
print(f"RAM: {get_ram():.2f} MB")
available_ram = psutil.virtual_memory().available / (1024**3) # in GB
print(f"Freier RAM: {available_ram:.2f} GB")

In [ ]:
# Der Polars Lazy-Frame
lf = pl.scan_parquet(DATA)

# EDA-Sample: ~5% des Gesamtdatensatzes als Pandas DataFrame für wgnd.inspect
#
# Strategie: gather_every(2) + sample(fraction=0.1)
# → gather_every(2) läuft lazy (vor collect) und halbiert den Datensatz systematisch,
#   d.h. jede zweite Zeile über den gesamten Zeitraum — zeitliche Abdeckung bleibt erhalten.
# → sample(fraction=0.1) zieht danach zufällig 10% aus dieser Hälfte.
# → Ergebnis: ~5% des Gesamtdatensatzes, deterministisch via SEED.
#
# Warum kein stratifiziertes Sampling?
# Für die EDA robust genug: canceled (4.5%) und alle 18 Linien sind bei 5% (~4.5 Mio. Zeilen)
# mit hoher Wahrscheinlichkeit gut vertreten. Stratifizierung nach line_name ist für die
# Modellierungsphase vorgemerkt (BACKLOG #3) — dort kritischer als hier.


df_eda = (
    lf
    .gather_every(2)
    .collect()
    .sample(fraction=0.1, seed=SEED)
    .to_pandas()
)

# Analoge Polars-Variante des Samples (für Polars-spezifische Operationen)
lf_eda = (
    lf
    .gather_every(2)
    .collect()
    .sample(fraction=0.1, seed=SEED)
)

# Bereinigte Delay-Variante: |delay| > 3.600s entfernt — für Verteilungs-Visualisierungen
df_delays_clean = df_eda[df_eda["arrival_delay"].abs() <= 3600].copy()

#### Note on Large Dataset Size





**Best Practice** 
Das Grundprinzip, bei einer EDA mit großen Datensätzen, heißt "**Lazy first**, **Sample second**, **Full last**" — arbeiten in drei Stufen:

1. Verwendung von **Polars** statt Pandas, um mit der **Lazy-API** deskriptive Statistiken (Mean, Median, Verteilungen) üer den gesamten Datensatz zu berechnen.
2. Verwendung von **Samples** mit Polars für die visuelle Inspektion von Einzelfällen. Vergleich der Sample Mittelwerte mit den Mittelwerte des gesamten Datensatzes.
3. Die Reihenfolge bleibt, Meta-Daten, Statistiken global, Visualisierung via Sample.

##### Sample Strategien


Zufallsauswahl nach Anzahl / 5k
```
df_sample = df.sample(n=5000)
```
Zufallsauswahl nach Anteil / 10%
```
df_sample = df.sample(fraction=0.1) 
```
Mit Zurücklegen
```
df_sample = df.sample(fraction=0.1, with_replacement=True)
```
Mit Reproduzierbarkeit
```
df_sample = df.sample(fraction=0.1, seed=42)
```
Systematische Auswahl / jedes 10te
```
df_sample = df.gather_every(10)
```
Kopf- bzw Fußzeilen
```
df_top = df.head(10)   
df_bottom = df.tail(10)
```
X zufällige Samples nach Gruppe
```
df_stratified = df.group_by("category").agg(
    pl.all().sample(n=100)
).explode(pl.all())

```

##### Übersicht der Sampling-Methoden in Polars

| Methode | Logik | Anwendungsfall | Performance |
| :--- | :--- | :--- | :--- |
| **`.sample(n/fraction)`** | Zufallsauswahl | Repräsentative EDA, Training von ML-Modellen | Mittel |
| **`.gather_every(n)`** | Jedes n-te Element | Zeitreihen, Sensordaten, sehr große Streams | Sehr schnell |
| **`.head(n)` / `.tail(n)`** | Erste / Letzte Zeilen | Schneller Check von Schema und Datentypen | Blitzschnell |
| **`.filter(condition)`** | Logische Bedingung | Fokus auf spezifische Teilpopulationen | Schnell |
| **`.unique()`** | Keine Duplikate | Analyse der Vielfalt von Kategorien | Schnell |
| **`group_by().sample()`** | Pro Gruppe n Zeilen | Stratified Sampling (z.B. bei Betrugserkennung) | Langsamer |

## Analysis



### Basic Statistical Analysis



> _Vollständige EDA-Übersicht mit wgnd.inspect._

In [ ]:
# --- Pandas with wgnd.inspect 

result = inspect(df_eda, sections=['memory', 'dimensions', 'dtypes', 'numeric', 'categorical'])

In [ ]:
# --- Polars insights

section_header('DIMENSIONS')

schema = lf.collect_schema()

# Anzahl der Spalten (sofort verfügbar)
print(f"Anzahl Spalten: {len(schema)}")

# Zeilenanzahl (erfordert einen schnellen Scan der Metadaten)
print(f"Anzahl Zeilen: {lf.select(pl.len()).collect().item()}")

# Null-Counts pro Spalte — aggregiert
print("Anzahl NaN:")
display(lf.select(pl.all().null_count()).collect())

section_header('DTYPES')

for col_name, dtype in schema.items():
    print(f"{col_name:<20} | Typ: {dtype}")

section_header('NUMERIC STATS')

# Umfassende Statistiken für alle Spalten
display( lf.describe())

section_header('CATEGORICAL STATS ')

# Helper-Funktion aus utils_polars.py
get_categorical_stats(lf_eda)


### Data Completeness



**Quality Check & Duplicates & Missing Values**

In [ ]:
# --- Pandas with wgnd.inspect_missing
# --- Und Missing Data Pattern  Heatmap  /// ca. 5 min. 

inspect_missing(df_eda)

In [ ]:
# --- Polars insights for missing values

section_header('Missing Values')


# Null-Counts pro Spalte — aggregiert
print("Anzahl NaN:")
display(lf.select(pl.all().null_count()).collect())

total_items = lf.select(pl.len()).collect().item()

result_missings = (
    lf
    .select(pl.all().null_count())
    .collect()
    .transpose(include_header=True, column_names=["null_count"])
    .with_columns(
        (pl.col("null_count") / total_items * 100).round(2).alias("pct")
    )
)

display(result_missings)


#### Missings — Schedule vs. Delay Asymmetrie

**Warum interessant?**  
`arrival_schedule` und `arrival_delay` sollten denselben Null-Anteil haben — wenn kein Zeitstempel vorhanden ist, kann auch kein Delay gemessen werden. Die Counts weichen aber ab: Schedule hat weniger Nulls als Delay. Das bedeutet: es gibt Fahrten, bei denen ein Zeitstempel registriert wurde, aber kein Delay-Wert ankam — typischerweise ein Übertragungsfehler im Sensor-/Reporting-System.

**Was ist auffällig?**  
| Spalte | Null-Count | Anteil |
|---|---|---|
| `arrival_schedule` | 120.477 | 0.13% |
| `arrival_delay` | 195.146 | 0.22% |
| `departure_schedule` | 120.477 | 0.13% |
| `departure_delay` | ~195.228 | 0.22% |

Differenz: ~74.669 Zeilen haben einen Schedule-Wert aber keinen Delay → Fahrt gemeldet, Messung nicht angekommen.

**Empfehlung Cleaning:**  
- Zeilen mit `arrival_schedule IS NULL` → rausfiltern (keine Zeitinformation, für Modell nutzlos)  
- Zeilen mit `arrival_schedule NOT NULL` aber `arrival_delay IS NULL` → für Ausfallanalyse behalten, für Delay-Modell herausfiltern (kein Zielwert vorhanden)

In [ ]:
# --- Finding C1: Missings in arrival/departure schedules and delays

section_header('Missings in schedules and delays')

# Asymmetrie zwischen Schedule- und Delay-Nullwerten prüfen
result_c1 = lf.select([
    pl.col("arrival_schedule").null_count().alias("arrival_schedule_null"),
    pl.col("arrival_delay").null_count().alias("arrival_delay_null"),
    pl.col("departure_schedule").null_count().alias("departure_schedule_null"),
    pl.col("departure_delay").null_count().alias("departure_delay_null"),
]).collect()

display(result_c1)

# Kernfrage: Wie viele Zeilen haben Schedule vorhanden, aber Delay fehlt?
result_c1_detail = lf.select([
    # Schedule null, Delay auch null
    ((pl.col("arrival_schedule").is_null()) & (pl.col("arrival_delay").is_null()))
        .sum().alias("beide_null"),
    # Schedule vorhanden, aber Delay fehlt → Übertragungsfehler
    ((pl.col("arrival_schedule").is_not_null()) & (pl.col("arrival_delay").is_null()))
        .sum().alias("schedule_ok_delay_null"),
    # Schedule fehlt, aber Delay vorhanden → unplausibler Sonderfall
    ((pl.col("arrival_schedule").is_null()) & (pl.col("arrival_delay").is_not_null()))
        .sum().alias("schedule_null_delay_ok"),
]).collect()

display(result_c1_detail)

total = lf.select(pl.len()).collect().item()

nutzlos = result_c1_detail["schedule_ok_delay_null"][0] + result_c1_detail["beide_null"][0]
log(f"\n>> Zeilen ohne verwertbaren Delay-Wert: {nutzlos:,}  ({nutzlos/total*100:.2f}% des Gesamtdatensatzes)")


#### Completeness Lesson: `is_windy` — 100% NaN across all 94M Rows

**Befund:** `wind_speed` ist vorhanden und plausibel befüllt. Das abgeleitete Flag `is_windy`
war jedoch über alle 94 Millionen Zeilen und alle 3 Jahre hinweg ausnahmslos `NaN` —
kein partielles Problem, sondern die Spalte wurde vom Meteo-Datenlieferanten schlicht nie befüllt.

**Konsequenz:** `is_windy` wird aus dem Feature-Set entfernt (→ F-WEAT-03). Als Ersatz bleibt
`wind_speed` (kontinuierlich) verfügbar und wird als Feature genutzt.

**Methodische Lektion:** Datenqualitätsprüfung muss für **jede einzelne Spalte** explizit
durchgeführt werden — nicht nur für offensichtliche Verdächtige. Eine Null-Rate von 100%
ist kein Fehler den man "sieht"; man muss systematisch danach suchen. Das hat die
8-Checks-Validierungspipeline in `zh-tram-data` ausgelöst:
Schema · Abdeckung · Wertebereiche · Nulls · Join-Qualität.


### Data Integrity



**Invalid Data Detection & Plausibility Check**

#### Extreme Delay-Werte (-29.892s / +34.685s)

**Warum interessant?**  
Verspätungen von -8.3 Stunden (zu früh) oder +9.6 Stunden (zu spät) sind physikalisch nicht plausibel. Ein Tram fährt im Taktbetrieb — Abweichungen über ±1 Stunde deuten fast immer auf Datenfehler hin (falsches Datum eingetragen, Systemfehler bei der Zeitstempelung). Besonders auffällig: der identische Maximalwert bei `arrival_delay` und `departure_delay` (34.685s) — das ist kein Zufall, sondern ein systematischer Fehler in einem spezifischen Datensatz.

**Was ist auffällig?**  
| Kennzahl | arrival_delay | departure_delay |
|---|---|---|
| Min | -29.892s = **-8.3h** | -29.876s = **-8.3h** |
| Max | +34.685s = **+9.6h** | +34.685s = **+9.6h** |
| Identischer Maxwert | ✅ | ✅ |

Faustformel: Alles über ±3.600s (±1h) ist unplausibel für regulären Tramverkehr.

**Empfehlung Cleaning:**  
- Zeilen mit `|delay| > 3.600s` (1h) → rausfiltern oder als `unreliable` flaggen  
- Alternativ: härtere Grenze `|delay| > 1.800s` (30 Min) prüfen — nach Betrachtung der Verteilung entscheiden

In [ ]:
# --- Finding I1: Extreme Delay-Werte

section_header('Extreme Delay-Werte')
print() 

# Wie viele Zeilen überschreiten plausible Grenzen?
thresholds = [600, 1800, 3600]  # 10 Min, 30 Min, 1 Stunde in Sekunden

for t in thresholds:
    n = lf.filter(
        (pl.col("arrival_delay").abs() > t) | (pl.col("departure_delay").abs() > t)
    ).select(pl.len()).collect().item()
    total = lf.select(pl.len()).collect().item()
    log(f">> delay >> {t:>5}s ({t//60:>2} Min): {n:>8,} Zeilen  ({n/total*100:.3f}%)")

print()

# Die extremsten Fälle anschauen — was steckt dahinter?
extreme = (
    lf
    .filter(pl.col("arrival_delay").abs() > 3600)
    .select(["operating_date", "line_name", "stop_name", "arrival_delay", "departure_delay", "canceled"])
    .sort("arrival_delay")
    .collect()
)

print(f"Zeilen mit |arrival_delay| > 1h: {len(extreme):,}")
print("\nExtremste negative Werte (zu früh):")
display(extreme.head(5))
print("\nExtremste positive Werte (zu spät):")
display(extreme.tail(5))

#### BPUIC Outlier (max 859,600,701)

**Warum interessant?**  
`bpuic` (Betriebspunkt-Identifikation) ist eine strukturierte Haltstellen-ID mit definiertem Format. VBZ-Haltestellen liegen konsistent im Bereich 8.502.572–8.596.001 (7–8-stellig). Der Maximalwert 859.600.701 ist 9-stellig und ~100× größer als der 75%-Wert — klar außerhalb jedes plausiblen Bereichs. Die hohe Skewness (31.6) im bpuic wird fast ausschließlich durch diese Ausreißer erzeugt.

**Was ist auffällig?**  
| Kennzahl | Wert |
|---|---|
| 75%-Perzentil | 8.591.335 |
| Max | 859.600.701 |
| Faktor | ~100× |
| Skewness | 31.6 |

Erwarteter Bereich: `8.500.000 – 8.600.000`  
Anomalie-Schwelle: `> 100.000.000`

**Empfehlung Cleaning:**  
- Zeilen mit `bpuic > 100.000.000` → rausfiltern  
- Prüfen ob diese Zeilen mit bestimmten Linien oder Zeiträumen korrelieren (systematischer Fehler in der Quelle?)

In [ ]:
# --- Finding I2: bpuic-Ausreißer

section_header('BPUIC Aussreisser')

bpuic_threshold = 100_000_000

anomale = (
    lf
    .filter(pl.col("bpuic") > bpuic_threshold)
    .select(["operating_date", "line_name", "bpuic", "stop_name"])
    .collect()
)

total = lf.select(pl.len()).collect().item()


# Verteilung der anomalen bpuic-Werte
display(anomale.group_by("bpuic").agg(
    pl.len().alias("n"),
    pl.col("line_name").first().alias("linie"),
    pl.col("operating_date").min().alias("datum_min"),
    pl.col("operating_date").max().alias("datum_max"),
).sort("n", descending=True))

# Normaler Bereich zur Kontrolle
normal_range = lf.select([
    pl.col("bpuic").filter(pl.col("bpuic") <= bpuic_threshold).min().alias("bpuic_min_normal"),
    pl.col("bpuic").filter(pl.col("bpuic") <= bpuic_threshold).max().alias("bpuic_max_normal"),
]).collect()

log(f"\n>> Zeilen mit bpuic > {bpuic_threshold:,}: {len(anomale):,}  ({len(anomale)/total*100:.4f}%)")
log(f"\n>> Normaler bpuic-Bereich: {normal_range['bpuic_min_normal'][0]:,} – {normal_range['bpuic_max_normal'][0]:,}")

#### Meteo Gaps (Missing Measurements, Rolling Window)

**Warum interessant?**  
Wetterdaten werden stündlich gemessen. Messausfälle entstehen durch Sensor-Neustarts, Wartung oder Übertragungsfehler — sie sind kurz (einzelne Stunden) und zeitlich klumpend. Da der Join auf `floor(arrival_schedule, '1h')` basiert, fallen alle Tramfahrten einer ausgefallenen Stunde gleichzeitig raus. Ein Rolling Mean über ±2 Nachbarstunden füllt diese Lücken zuverlässig.

**Was ist auffällig?**  
| Spalte | Null-Count (Gesamtdatensatz) | Anteil |
|---|---|---|
| `temperature` | ~315.000 | ~0.35% |
| `humidity` | ~315.000 | ~0.35% |
| `rain_duration` | ~237.000 | ~0.26% |
| `wind_speed` | ~237.000 | ~0.27% |
| `global_radiation` | ~258.000 | ~0.29% |
| `flood_intensity` | ~129.000 | ~0.14% |

Alle Meteo-Spalten haben ähnliche Null-Counts → Ausfälle betreffen immer alle Sensoren gleichzeitig (Stationsausfall, nicht einzelne Sensoren).

**Empfehlung Cleaning:**  
- Rolling Mean über `window_size=5` (±2 Stunden) pro Tag: `pl.col("temperature").rolling_mean(5)`  
- Wenn keine Nachbarn vorhanden (Beginn/Ende): Tages-Median als Fallback  
- `flood_intensity` separat behandeln — Rolling Mean ergibt hier keinen Sinn (Ereignis-Indikator), stattdessen `fill_null(0)`

In [ ]:
# --- Finding I3: Meteo-Lücken — Muster und Umfang prüfen

section_header('Meteo-Missings')

meteo_cols = ["temperature", "humidity", "rain_duration", "precipitation", "wind_speed", "global_radiation", "flood_intensity"]

# Null-Counts im Gesamtdatensatz
null_counts = lf.select([pl.col(c).null_count().alias(c) for c in meteo_cols]).collect()
total = lf.select(pl.len()).collect().item()

for col in meteo_cols:
    n = null_counts[col][0]
    print(f"{col:<20} null: {n:>8,}  ({n/total*100:.2f}%)")

print()

# Sind die Ausfälle zeitlich klumpend? → Stunden mit vielen gleichzeitigen Null-Werten
lücken_muster = (
    lf
    .filter(pl.col("temperature").is_null())
    .with_columns(
        pl.col("arrival_schedule").dt.truncate("1h").alias("stunde")
    )
    .group_by("stunde")
    .agg(pl.len().alias("n_fahrten_ohne_meteo"))
    .sort("stunde")
    .collect()
)

print(f"Stunden mit fehlenden Temperaturwerten: {len(lücken_muster):,}")
print("Erste 10 Lücken-Stunden:")
display(lücken_muster.head(10))

#### Humidity > 100% (Sensor-Kalibrierungsfehler)

**Warum interessant?**  
Relative Luftfeuchtigkeit ist physikalisch auf 0–100% begrenzt. Werte leicht über 100% (bis ~102%) sind ein bekanntes Kalibrierungsproblem bei Wetterstationen und kein echter Messfehler — aber sie müssen vor dem Modelltraining gekappt werden, da kein ML-Modell mit >100% Luftfeuchtigkeit umgehen kann.

**Was ist auffällig?**  
| Kennzahl | Wert |
|---|---|
| Max `humidity` (Sample) | 101.37% |
| Erwarteter Wertebereich | 0 – 100% |
| Wahrscheinliche Ursache | Sensor-Kalibrierungsdrift |

Werte unter 0% wären ebenfalls anomal — auch das wird mitgeprüft.

**Empfehlung Cleaning:**  
- `humidity` auf `[0, 100]` cappen: `pl.col("humidity").clip(0, 100)`  
- Kein Rausfiltern der Zeilen nötig — nur der Wert selbst wird korrigiert

In [ ]:
# --- Finding I4: Humidity > 100% und weitere Wertebereichs-Checks

section_header('Humidity Value Check')

# Humidity-Anomalien
result_humidity = lf.select([
    pl.col("humidity").min().alias("min"),
    pl.col("humidity").max().alias("max"),
    (pl.col("humidity") > 100).sum().alias("n_über_100"),
    (pl.col("humidity") < 0).sum().alias("n_unter_0"),
    pl.col("humidity").filter(pl.col("humidity") > 100).mean().alias("mean_über_100"),
]).collect()

display(result_humidity)
total = lf.select(pl.len()).collect().item()

print()

# Plausibilitäts-Check weiterer physikalischer Grenzen
print("Weitere Wertebereichs-Checks:")
checks = lf.select([
    (pl.col("rain_duration") > 60).sum().alias("rain_duration_über_60min"),   # max ist 60 min/h
    (pl.col("rain_duration") < 0).sum().alias("rain_duration_unter_0"),
    (pl.col("wind_speed") < 0).sum().alias("wind_speed_unter_0"),
    (pl.col("temperature") < -30).sum().alias("temperature_unter_minus30"),   # für Zürich unplausibel
    (pl.col("temperature") > 45).sum().alias("temperature_über_45"),          # für Zürich unplausibel
    (pl.col("district_nr") < 1).sum().alias("district_unter_1"),              # gültig: 1-12
    (pl.col("district_nr") > 12).sum().alias("district_über_12"),
]).collect()

display(checks)

log(f">> Anteil > 100%: {result_humidity['n_über_100'][0]:,}  ({result_humidity['n_über_100'][0]/total*100:.3f}%)")


#### Canceled Trips (~4.5% der Fahrten)

**Warum interessant?**  
Ausgefallene Fahrten (`canceled=True`) sind der extremste Verspätungsfall — keine Verzögerung, sondern kompletter Ausfall. Sie wurden im Wrangling bewusst behalten (worst-case für das Modell). Wichtig zu klären: Haben canceled trips sinnvolle Delay-Werte, oder sind die Delays bei Ausfall null? Das bestimmt wie wir sie im Modell behandeln.

**Was ist auffällig?**  
| Kennzahl | Wert |
|---|---|
| Anteil canceled (5%-Sample) | ~4.52% |
| Hochrechnung Gesamtdatensatz | ~4.0 Mio. Fahrten |
| Häufigste Linie mit Ausfällen | noch zu prüfen |

Frage: Haben canceled trips `arrival_delay = null` (Fahrt kam nicht → keine Messung) oder einen sehr hohen Delay-Wert?

**Empfehlung Cleaning:**  
- `canceled=True` Zeilen **behalten** — unverzichtbar für Modell  
- Als separate Gruppe oder als Feature `is_canceled` (0/1) kodieren  
- Delay-Werte bei canceled=True prüfen: wenn null → für Delay-Vorhersage herausfiltern, für Ausfallmodell behalten

In [ ]:
# --- Finding I5: Canceled Trips

section_header('Canceld Trips Analysis')

# Anteil und Verteilung
result_canceled = (
    lf
    .group_by("canceled")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct"))
    .sort("canceled")
    .collect()
)
display(result_canceled)

# Kernfrage: Haben canceled trips Delay-Werte?
result_canceled_delays = (
    lf
    .filter(pl.col("canceled"))
    .select([
        pl.len().alias("total_canceled"),
        pl.col("arrival_delay").null_count().alias("arrival_delay_null"),
        pl.col("departure_delay").null_count().alias("departure_delay_null"),
        pl.col("arrival_delay").mean().alias("mean_arrival_delay"),
        pl.col("arrival_delay").median().alias("median_arrival_delay"),
    ])
    .collect()
)

print("\nCanceled trips — Delay-Analyse:")
display(result_canceled_delays)

# Ausfälle nach Linie — welche Linie hat die meisten?
print("\nAusfälle pro Linie:")
display(
    lf
    .filter(pl.col("canceled"))
    .group_by("line_name")
    .agg(pl.len().alias("n_ausfaelle"))
    .sort("n_ausfaelle", descending=True)
    .collect()
)

### Data Distribution



**Numerical Features & Categorical Features & Bivariate Analysis**

In [ ]:

result = inspect(df_eda, sections=[ 'numeric'])

#### Numerische Features — Rohe Verteilung

In [ ]:
from wgnd.viz import grid_histplot
from wgnd.core.config import cfg

section_header('Numerische Verteilungen — Rohdaten')

# Delays — bereinigt (|delay| ≤ 3.600s) für saubere Skalierung; mit canceled als Hue
fig, _ = grid_histplot(
    df_delays_clean,
    columns=["arrival_delay", "departure_delay"],
    hue="canceled",
    multiple="layer",
    n_cols=2,
    figsize_per_plot=(7, 4),
)
plt.suptitle("Delay-Features — bereinigt (|delay| ≤ 3.600s, orange = canceled)",
             fontsize=13, color=cfg.CHART_TITLE, x=0.02, ha="left", y=1.02)
plt.show()

# Meteo-Features
fig, _ = grid_histplot(
    df_eda,
    columns=["temperature", "precipitation", "rain_duration",
             "wind_speed", "global_radiation", "humidity"],
    n_cols=3,
    figsize_per_plot=(6, 3),
)
plt.suptitle("Meteo-Features — Rohdaten",
             fontsize=13, color=cfg.CHART_TITLE, x=0.02, ha="left", y=1.02)
plt.show()

#### Kategorische Features — Verteilung

In [ ]:
from wgnd.viz import bar

section_header('Kategorische Verteilungen')

# Fahrten pro Linie
line_counts = df_eda["line_name"].value_counts().sort_values()
fig, ax = bar(line_counts, orient="h", title="Fahrten pro Linie (Sample)")
plt.show()

# Fahrten pro Stadtdistrikt (ohne null)
district_counts = df_eda["district_name"].dropna().value_counts().sort_values()
fig, ax = bar(district_counts, orient="h", title="Fahrten pro Stadtdistrikt (Sample, ohne außerhalb)")
plt.show()

# Canceled-Anteil pro Linie — stacked bar
line_cancel = (
    df_eda.groupby("line_name", observed=True)["canceled"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("pct")
    .reset_index()
)
canceled_pct = (
    line_cancel[line_cancel["canceled"] == True]
    .set_index("line_name")["pct"]
    .sort_values()
)
fig, ax = bar(
    canceled_pct, orient="h",
    title="Canceled-Anteil pro Linie (%)",
    ref_val=canceled_pct.mean(), ref_label="Ø"
)
plt.show()

#### Beurteilung der Verteilungen

| Priorität | ID | Befund | Kategorie | Empfehlung | Geklärt? |
|:---|:---:|:---|:---:|:---|:---:|
| **Kritisch** | I1 | Extreme Delays: departure_delay Skewness 42.9, arrival_delay 38.6 — Ausreißer verzerren Modell stark | Integrity | Outlier-Bereinigung via IQR 3× als eigener Cleaning-Schritt | ✓ |
| **Kritisch** | C1 | arrival_schedule/delay Asymmetrie — 74.669 Zeilen: Schedule vorhanden, Delay null; 0 umgekehrt; 195.146 Zeilen (0.22%) ohne verwertbaren Delay-Wert | Completeness | Zeilen mit Schedule-ok aber Delay-null ausschließen oder imputieren | |
| **Hoch** | I4 | `relative_humidity` > 100 % — physikalisch unmöglich | Integrity | Clip auf [0, 100] im Cleaning-Schritt | |
| **Hoch** | I2 | `bpuic` kein numerisches Feature: 91.137 Zeilen (0.10%) außerhalb Normalbereich 8.502.572–8.596.007 — strukturierter ID-Typ | Integrity | Nicht als Float-Feature ins Modell; als kategoriales Label oder Lookup-Key behandeln | ✓ |
| **Mittel** | I3 | Negative Delays — Züge kommen früher als geplant an | Integrity | Plausibel und erwünscht; Wertebereich dokumentieren, kein Clip nötig | ✓ |
| **Mittel** | C3 | `district_nr` Nulls — nicht alle Haltestellen einer Stadtdistrikt zugeordnet | Completeness | Plausibel (Stadtgrenze / GTFS-Mapping); separat dokumentieren | ✓ |
| **Niedrig** | I5 | Zero-Inflated: `precipitation` (Skewness 11.6), `flood_intensity` (14.6), `rain_duration` (2.68) — dominiert durch Nullen | Integrity | Binäres Flag (`hat_regen`) + kontinuierlicher Wert; nicht log-transformieren | ✓ |
| **Niedrig** | C2 | `event_type` / `event_size` fast überall null | Completeness | Plausibel — Normalfall ist kein Event; als Dummy-Feature kodieren | ✓ |
| **Niedrig** | C4 | Duplikate: 1.72 % doppelte Zeilen im Rohdatensatz | Completeness | `distinct()` im Cleaning-Schritt; Ursache unklar (doppelte GTFS-Einträge?) | |

### Data Relationships



**Correlations**

In [ ]:
import matplotlib.pyplot as plt
from wgnd.core.config import cfg

# Session-Patch: installed wgnd version erwartet noch PALETTE_DIVERGENT
# Permanent-Fix benötigt uv reinstall — hier temporäre Lösung für diese Session
_prg = plt.get_cmap(cfg.PALETTE_DIV)
cfg.PALETTE_DIVERGENT = [_prg(i / 6) for i in range(7)]
print("✅ Session-Patch aktiv")

In [ ]:
inspect_correlations(df_eda)
# inspect_correlations(df_eda, target='your_target_col', show_pairplot=True)

#### Korrelations-Findings

| # | Finding | Paare | r | Interpretation | Konsequenz |
|:---|:---|:---|:---:|:---|:---|
| R1 | Delay-Selbstkorrelation | `arrival_delay` ↔ `departure_delay` | 0.95 | Selbe Verspätung setzt sich fort — logisch | Nur eine der beiden als Zielvariable verwenden |
| R2 | Wetter→Delay: schwache lineare Signale | `rain_duration` / `precipitation` ↔ `arrival_delay` | 0.03 | Kein linearer Zusammenhang — Schwellenwert-Effekte wahrscheinlich (Starkregen ≠ leichter Regen) | Baummodell (XGBoost) statt linearer Regression — kann Threshold-Effekte lernen |
| R3 | Events sind saisonal | `event_size` ↔ `temperature` | 0.16 | Große Events finden im Sommer statt (Street Parade, Open Air) — beide Features indirekt saisonal | Nicht gemeinsam ins Modell ohne Saisonalität zu kontrollieren; Monats-Feature erwägen |
| R4 | Überflutung durch Intensität, nicht Dauer | `flood_intensity` ↔ `precipitation` | 0.15 | Stärkster Wetter-Korrelationswert — Intensität entscheidender als Regendauer (r=0.10) | Zero-Inflated behandeln: binäres Flag `hat_flut` + kontinuierlicher Wert |
| R5 | Geografische Multikollinearität | `stop_lat` ↔ `district_nr` | 0.68 | Geografisch erwartet, kein Informationsgewinn | `district_nr` und `stop_lat/lon` nicht gemeinsam ins Modell |

> **Modellierungs-Fazit:** Schwache lineare Korrelationen zwischen Wetter und Delay sind kein Zeichen für Irrelevanz — sie zeigen Nicht-Linearität. XGBoost kann Schwellenwert-Effekte (Starkregen, Extremtemperatur) erkennen, wo Pearson-r versagt. Feature-Engineering-Ideen (Extremwert-Flags, Interaktionen Regen × Distrikt) → Phase 3.

### Outlier Detection

In [ ]:
inspect_outliers(df_eda)
# inspect_outlier_detail(df_eda, 'your_col')

#### Delay-Features — Outlier Detail (IQR 1.5× und 3×)

In [ ]:
section_header('Delay Outlier — Vor / Nach Bereinigung')
log(f"Roh: {len(df_eda):,} Zeilen  →  Bereinigt: {len(df_delays_clean):,} Zeilen  "
    f"({len(df_eda) - len(df_delays_clean):,} entfernt, "
    f"{(len(df_eda) - len(df_delays_clean)) / len(df_eda) * 100:.3f}%)")

print("\n── arrival_delay  ROH ──")
inspect_outlier_detail(df_eda, "arrival_delay")

print("\n── arrival_delay  BEREINIGT (|delay| ≤ 3.600s) ──")
inspect_outlier_detail(df_delays_clean, "arrival_delay")

print("\n── departure_delay  ROH ──")
inspect_outlier_detail(df_eda, "departure_delay")

print("\n── departure_delay  BEREINIGT ──")
inspect_outlier_detail(df_delays_clean, "departure_delay")

#### Meteo-Features — Outlier Detail (precipitation, wind_speed, temperature)

In [ ]:
from wgnd.core.theme import mpl_style
from wgnd.core.config import cfg

section_header('Meteo Outlier Detail')

# wind_speed und temperature: inspect_outlier_detail reicht
inspect_outlier_detail(df_eda, "wind_speed")
inspect_outlier_detail(df_eda, "temperature")

# precipitation: Zero-Inflation + Log-Skala Gegenüberstellung
section_header('Precipitation — Normal vs. Log-Skala')

style = mpl_style()
data      = df_eda["precipitation"].dropna()
data_nz   = data[data > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Links: Normal — Zero-Inflation sichtbar
axes[0].hist(data, bins=60, color=cfg.COLOR_NEGATIVE, alpha=0.85, edgecolor="none")
axes[0].set_title("Normal-Skala — Zero-Inflation dominiert", **style["title"])
axes[0].set_xlabel("precipitation (mm)", **style["label"])
axes[0].set_ylabel("Häufigkeit", **style["label"])
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)

# Rechts: Log-Skala, nur Regen-Stunden
axes[1].hist(data_nz, bins=60, color=cfg.ACTIVE_PALETTE[0], alpha=0.85, edgecolor="none")
axes[1].set_xscale("log")
axes[1].set_title(f"Log-Skala — nur Regen-Stunden (n={len(data_nz):,})", **style["title"])
axes[1].set_xlabel("precipitation (mm, log)", **style["label"])
axes[1].set_ylabel("Häufigkeit", **style["label"])
axes[1].axvline(data_nz.median(), color=cfg.COLOR_SIGNAL, linewidth=1.5,
                linestyle="--", label=f"Median: {data_nz.median():.1f}mm")
axes[1].axvline(data_nz.mean(), color=cfg.COLOR_POSITIVE, linewidth=1.5,
                linestyle="--", label=f"Mean: {data_nz.mean():.1f}mm")
axes[1].legend(fontsize=10)
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)

plt.suptitle("precipitation — Zero-Inflation und echte Regenverteilung",
             fontsize=14, color=cfg.CHART_TITLE, ha="left", x=0.02, y=1.02)
plt.tight_layout()
plt.show()

success(f"Regen-Stunden: {len(data_nz):,} von {len(data):,} "
        f"({len(data_nz)/len(data)*100:.1f}%) — "
        f"{100 - len(data_nz)/len(data)*100:.1f}% sind 0")

#### Outlier-Findings

| # | Feature | Skewness | IQR 1.5× Outlier | IQR 3× Outlier | Muster | Vorgehen |
|:---|:---|:---:|:---|:---|:---|:---|
| O1 | `arrival_delay` | 38.6 | hoch | 0.005% (4.707) | Langer rechter Schwanz, extreme Einzelwerte | Rausfiltern `\|delay\| > 3.600s` |
| O2 | `departure_delay` | 42.9 | hoch | 0.005% (4.707) | Identisch zu O1 — systematischer Fehler | Gleiche Grenze wie O1 |
| O3 | `precipitation` | 11.6 | sehr hoch | — | Zero-Inflated: IQR ≈ 0, jeder Regen = statistischer Ausreißer | Kein Clip — binäres Flag `hat_regen` + Wert behalten |
| O4 | `wind_speed` | 1.34 | moderat | — | Leichter Skew, Sturmtage als Ausreißer plausibel | Kein Clip — Extremwerte sind real und modellrelevant |
| O5 | `temperature` | ~0 | minimal | — | Saubere Normalverteilung, keine echten Ausreißer | Keine Bereinigung nötig |

### Features Inspection



**Engineering Inspection**

In [ ]:
# Feature Engineering — erste Ideen aus der EDA (Umsetzung in 02_preparation.ipynb)
#
# Zeitfeatures
# stunde         : arrival_schedule.dt.hour          → Tagesrhythmus
# wochentag      : arrival_schedule.dt.weekday       → Mo=0, So=6
# monat          : arrival_schedule.dt.month         → Saisonalität (R3: Events saisonal)
# ist_wochenende : wochentag >= 5
# ist_hvz        : stunde in [7,8,9,17,18,19]        → Hauptverkehrszeit
#
# Binäre Extremwert-Flags (aus R2/R4/O3)
# hat_regen      : precipitation > 0
# hat_starkregen : precipitation > 5.0               → Schwellenwert aus Viz (Log-Skala)
# hat_flut       : flood_intensity > 0
# hat_wind       : wind_speed > 40                   → nach Outlier-Analyse bestimmen
#
# Kategoriale Encodings
# line_name      : Label-Encoding oder Target-Encoding
# district_name  : Label-Encoding (inkl. Kategorie "ausserhalb" für Nulls)
# event_size     : Ordinal — kein Event=0, klein=1, mittel=2, groß=3
# is_canceled    : bereits vorhanden als bool → zu int konvertieren
#
# Interaktionen (für Phase 4, nach Feature Importance)
# hat_regen × district_nr → Welche Stadtkreise leiden bei Regen stärker?
# ist_hvz × line_name     → Welche Linien sind in der HVZ am instabilsten?

print("Feature Engineering — erste Ideen notiert. Umsetzung in 02_preparation.ipynb.")

#### Feature Ideas — Overview

| Kategorie | Feature | Basis-Spalte | Begründung |
|:---|:---|:---|:---|
| **Zeit** | `stunde` | `arrival_schedule` | Tagesrhythmus — Rush Hour vs. Randzeiten |
| **Zeit** | `wochentag` | `arrival_schedule` | Werktagsmuster, Wochenend-Effekte |
| **Zeit** | `monat` | `arrival_schedule` | Saisonalität (R3: Events korreliert mit Temp.) |
| **Zeit** | `ist_wochenende` | `wochentag` | Binäres Flag — kompakter als Ordinal |
| **Zeit** | `ist_hvz` | `stunde` | Hauptverkehrszeit 7–9h, 17–19h |
| **Wetter** | `hat_regen` | `precipitation > 0` | Zero-Inflation auflösen (O3) |
| **Wetter** | `hat_starkregen` | `precipitation > 5mm` | Schwellenwert-Effekt aus Viz (R2) |
| **Wetter** | `hat_flut` | `flood_intensity > 0` | Stärkste Wetter-Korrelation r=0.15 (R4) |
| **Wetter** | `hat_wind` | `wind_speed > X` | Schwellenwert nach Outlier-Analyse |
| **Kategorial** | `line_name` enc. | `line_name` | Label- oder Target-Encoding |
| **Kategorial** | `district_name` enc. | `district_name` | inkl. Kategorie `"ausserhalb"` für Nulls |
| **Kategorial** | `event_size` enc. | `event_size` | Ordinal 0–3 (kein Event bis groß) |
| **Kategorial** | `is_canceled` | `canceled` | bool → int (0/1) |
| **Interaktion** | `regen × district` | Kombination | Räumliche Sensitivität bei Regen |
| **Interaktion** | `hvz × linie` | Kombination | Linienstabilität in der Hauptverkehrszeit |

## Key Findings & Next Steps


### Konsolidierte Findings — nach Topic

| Topic | Befund | Empfehlung | Vor Split? |
|:---|:---|:---|:---:|
| **Delay** | Extreme Werte ±8.3h — physikalisch nicht plausibel | Rausfiltern `\|delay\| > 3.600s` | ✓ |
| **Delay** | 74.669 Zeilen: Schedule vorhanden, aber kein Delay | Herausfiltern für Delay-Modell | ✓ |
| **Delay** | `arrival_delay` ↔ `departure_delay` r=0.95 | Nur `arrival_delay` als Zielvariable | — |
| **Delay** | Skewness 38–43 — non-linear, long tail | XGBoost statt lineares Modell | — |
| **Canceled** | 4.5% ausgefallene Fahrten, haben Delay-Werte | Feature `is_canceled`; für Delay-Modell optional herausfiltern | ✓ |
| **BPUIC** | 0.10% anomale IDs > 100.000.000 | Rausfiltern | ✓ |
| **BPUIC** | Nicht als numerisches Feature geeignet | Als kategoriales Label oder Lookup-Key | ✓ |
| **Meteo** | Stündliche Messausfälle (~0.14–0.35%), zeitlich klumpend | Rolling Mean ±2h; `flood_intensity` → `fill_null(0)` | Nach Split |
| **Meteo** | `humidity` > 100% — Sensor-Kalibrierungsdrift | `.clip(0, 100)` | ✓ |
| **Meteo** | `precipitation` zero-inflated (Skewness 11.6) | Flag `hat_regen` + kontinuierlicher Wert behalten | Nach Split |
| **Meteo** | Wetter→Delay: r max 0.03 linear | Schwellenwert-Effekte → XGBoost; Extremwert-Flags als Features | — |
| **Events** | 78.5% null — Normalfall kein Event | `null` → Kategorie `"kein_event"` | ✓ |
| **Events** | Events korreliert mit Temperatur r=0.16 — beide saisonal | Monats-Feature hinzufügen | — |
| **District** | 6.87% null — Haltestellen außerhalb Stadtgebiet | `null` → Kategorie `"ausserhalb"` | ✓ |
| **District** | `stop_lat` ↔ `district_nr` r=0.68 — Multikollinearität | Nicht beide gemeinsam ins Modell | — |
| **Datenqualität** | 1.72% Duplikate | `distinct()` im Cleaning | ✓ |

> **Vor Split?** — ✓ = strukturelles Cleaning (Domainregeln, keine statistischen Parameter → kein Leakage-Risiko)  
> **Nach Split** = statistisch abgeleitete Werte (Rolling Mean, IQR-Grenzen, Encoding) → nur auf Trainingsdaten fitten, auf Testdaten anwenden

### Offene Fragen — alle in der EDA beantwortet

| # | Frage | Antwort |
|---|---|---|
| F1 | Wie viele Zeilen: Schedule ok, aber Delay null? | ~74.669 Zeilen (0.08%) → Herausfiltern |
| F2 | Wie viele Zeilen liegen über ±3.600s? | ~4.707 Zeilen (0.005%) → Rausfiltern |
| F3 | Wie viele anomale BPUICs, welche Linien? | ~91.137 Zeilen (0.10%), mehrere Linien → Rausfiltern |
| F4 | Meteo-Lücken zeitlich klumpend oder zufällig? | Klumpend — Stationsausfälle, alle Sensoren gleichzeitig |
| F5 | Wie viele Humidity-Werte > 100%? | < 5.000 Zeilen — Clip auf 100 reicht |
| F6 | Haben canceled trips Delay-Werte oder sind sie null? | Haben Delay-Werte → als Feature `is_canceled` behalten |

### Prognose der Datensatz-Reduktion nach dem Cleaning

| Cleaning-Schritt | Betroffene Zeilen (ca.) | Anteil | Timing |
|:---|---:|:---:|:---|
| Duplikate entfernen | ~1.500.000 | 1.72% | Vor Split |
| BPUIC > 100.000.000 rausfiltern | ~91.100 | 0.10% | Vor Split |
| Schedule ok, aber Delay null | ~74.700 | 0.08% | Vor Split |
| Extreme Delays \|delay\| > 3.600s | ~4.700 | < 0.01% | Vor Split |
| Humidity clip auf [0, 100] | < 5.000 | < 0.01% | Vor Split |
| Meteo Rolling Mean ±2h (Imputation) | ~130k–315k je Spalte | 0.14–0.35% | Nach Split |
| **Gesamt-Reduktion (strukturell)** | **~1,7 Mio.** | **~2%** | — |

> Nach dem strukturellen Cleaning verbleiben schätzungsweise **~86–87 Mio. Zeilen**.  
> 2025 wird als Test-Jahr reserviert (~15–18 Mio. Zeilen) — Trainingsdatensatz: **~68–72 Mio. Zeilen**.